In [0]:
from pyspark.sql import functions as F

CATALOG = "edw_migration"
SCHEMA = "silver_edw"
VOLUME_PATH = "/Volumes/edw_migration/bronze_fusion/raw_files"

# ---------------------------------------------------------------
# ODI Step 1: PKG_WD_REVENUE_TYPE_DIM_STG_HDD
# ---------------------------------------------------------------
revenue_type_src = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .option("nullValue", "NULL")
    .csv(f"{VOLUME_PATH}/WD_REVENUE_TYPE_DIM_STG_SRC.csv")
)

revenue_type_df = (
    revenue_type_src
    .withColumn("DELETE_FLG", F.lit("N"))
    .withColumn("CREATE_DT", F.current_timestamp())
    .withColumn("UPDATE_DT", F.current_timestamp())
    .withColumn("INTEGRATION_ID", F.col("GM_CATEGORY3_NM"))
    .select("REV_TYP_NM", "REV_SUB_TYP_NM", "GM_CATEGORY1_NM", "GM_CATEGORY2_NM",
            "DELETE_FLG", "CREATE_DT", "UPDATE_DT", "INTEGRATION_ID", "GM_CATEGORY3_NM")
)
revenue_type_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.wd_revenue_type_dim_stg")
print(f"wd_revenue_type_dim_stg loaded: {revenue_type_df.count()} rows")

# ---------------------------------------------------------------
# ODI Step 2: PKG_WD_LOOKUP_HDD — straight truncate & load
# ---------------------------------------------------------------
lookup_df = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .option("nullValue", "NULL")
    .csv(f"{VOLUME_PATH}/WD_LOOKUP_HDD_SRC.csv")
)
lookup_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.wd_lookup_hdd")
print(f"wd_lookup_hdd loaded: {lookup_df.count()} rows")

# ---------------------------------------------------------------
# ODI Step 3: PKG_WD_BUS_RULES_HDD — straight truncate & load
# NOTE: GL_ACCOUNT stays a string — matches this table's existing column type
# ---------------------------------------------------------------
bus_rules_df = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .option("nullValue", "NULL")
    .csv(f"{VOLUME_PATH}/WD_BUS_RULES_HDD_SRC.csv")
)
bus_rules_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.wd_bus_rules_hdd")
print(f"wd_bus_rules_hdd loaded: {bus_rules_df.count()} rows")